(Molecular_Workbench)=
# Exploring molecular systems and recovering your work

Build a molecular scene step by step, control its regions, calculate hydrogen bonds on a short trajectory, and reopen the saved session. Run the cells in order in Jupyter. After a cell changes a view, return to that view's canvas to inspect the result.

The web documentation includes previews. Running the notebook gives you the live Python connection needed by Studio's selection and calculation controls.

:::{admonition} Experimental interaction backend
Interactions require a compatible MolSysMT build with the experimental Interactions and H5MSM APIs. Compatible published-package qualification is still pending. The examples use bundled data and require no downloads.
:::

In [1]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import molsysmt as msm
import molsysviewer as msv

protein = msm.systems["chicken villin HP35"]["1vii.pdb"]
caffeine = msm.systems["caffeine"]["caffeine.sdf"]
workspace = tempfile.TemporaryDirectory(prefix="msv-workbench-tutorial-")
output = Path(workspace.name)

## 1. Loading one source and adding another

Start with the protein. Whole represents the complete molecular system. Rotate it, zoom with the mouse wheel, and open the floating Studio card.

In [2]:
view_A = msv.MolSysView(height="520px")
view_A.load(protein, structure_indices=[0], label="Protein")
view_A

Add caffeine to the same view. Whole now contains both sources, and Regions contains one source region for each. You should see 596 protein atoms and 24 caffeine atoms.

This operation preserves the incoming coordinates. Preparing a meaningful protein–ligand complex requires an appropriate pose or alignment. Source-region identity lets you keep track of each input.

The first source defines the box policy. In this example the protein's box is retained, and caffeine has no box. MolSysMT reports that mismatch. Concatenation can also report discarded structural attributes such as occupancy and B factors; read these warnings before using the combined system for calculations.

In [3]:
view_A.load(caffeine, label="Caffeine")
pd.DataFrame(view_A.load_blocks)[["label", "n_atoms", "region_tag"]]

,label,n_atoms,region_tag
0,Protein,596,Protein
1,Caffeine,24,Caffeine


## 2. Controlling region visibility and representation

Use Hide and Show on the Caffeine card in **Studio → Regions → Saved Regions**. Caffeine should disappear while the protein remains, then return.

You can perform the same operations from Python. A region without its own representation controls its atoms in Whole.

In [4]:
protein_region = view_A.regions[view_A.load_blocks[0]["region_tag"]]
caffeine_region = view_A.regions[view_A.load_blocks[1]["region_tag"]]
caffeine_region.hide()

In [5]:
caffeine_region.show()

Give caffeine its own orange ball-and-stick representation. Hide still removes it from its own representation and from Whole. It does not hide atoms in another region's independent representation.

In [6]:
caffeine_region.set_representation("ball-and-stick", color="orange")
caffeine_region.hide()

Enabled controls whether the region participates in the scene. Disabling this hidden region releases its effect on Whole, so caffeine becomes visible through Whole again. Its representation and hidden request are retained.

In [7]:
caffeine_region.disable()

Enable it again: its saved hidden request takes effect, so caffeine disappears. Show then makes the enabled region visible.

In [8]:
caffeine_region.enable()

In [9]:
caffeine_region.show()

Python also offers `show_only()` to isolate a region. Restore the scene by showing both source regions and Whole. Saved Regions has no dedicated Show Only button.

In [10]:
caffeine_region.show_only()

In [11]:
protein_region.show()
caffeine_region.show()
view_A.whole.show()

The combined scene with an orange caffeine region looks like this:

<iframe src="../../../_static/views/workbench_sources.html" title="Protein and caffeine source regions" style="width:100%;height:520px;border:0;" loading="lazy"></iframe>

## 3. Loading the same sources together

Use the same `load()` method with `multiple=True` to identify independent molecular sources. Each source gets a region from the start. Whole contains the combined system.

This example combines single structures. Composition preserves source coordinates; it does not infer trajectory alignment. See {doc}`../molecular_system/index` for loading contracts.

In [12]:
view_B = msv.MolSysView(height="520px")
view_B.load(
    [protein, caffeine], multiple=True, structure_indices=[0],
    labels=["Protein", "Caffeine"],
)
view_B

In [13]:
xyz_A = msm.pyunitwizard.get_value(msm.get(view_A.molsys, coordinates=True), to_unit="nm")
xyz_B = msm.pyunitwizard.get_value(msm.get(view_B.molsys, coordinates=True), to_unit="nm")
np.testing.assert_array_equal(xyz_A, xyz_B)
pd.DataFrame(view_B.load_blocks)[["label", "n_atoms", "region_tag"]]

,label,n_atoms,region_tag
0,Protein,596,Protein
1,Caffeine,24,Caffeine


Both views have the same atoms and coordinates. Representations that you edited on view_A belong to that scene; view_B starts with its own scene configuration.

## 4. Calculating and filtering interactions on a trajectory

Load three real pentalanine structures in source order **[0, 8, 3]**. Their local indices in this view are **0, 1, 2**. Navigate between them with the canvas controls.

For an optional Studio calculation, choose **Calculate**, name it `studio-buch`, select H bonds / Buch, set the distance to **4 Å**, leave PBC off, and explicitly set **Calculate structures** to `all`. The default calculation scope is `current`. **Display structures** only filters saved results.

After calculation, Studio switches to Stored analysis. Select Calculate again to enter a new analysis name. Hide the Studio-generated visual set before displaying the Python example below.

In [14]:
view = msv.new_view(
    msm.systems["pentalanine"]["traj_pentalanine.h5msm"],
    structure_indices=[0, 8, 3], height="520px",
)
molsys = view.molsys
view

Calculate a named analysis on all three local structures. Buch uses the hydrogen–acceptor distance. The permissive **4 Å** cutoff supplies visible examples for this tutorial; choosing a scientifically appropriate hydrogen-bond criterion and cutoff depends on your system and question. It does not certify every displayed contact as a hydrogen bond.

The calculation stores the complete result in `molsys.interactions`. Creating a visual set is a separate step. Playback queries saved observations and does not recalculate them.

In [15]:
result = view.interactions.hbonds.get_buch_hbonds(
    name="buch", structure_indices="all",
    distance_threshold="4 angstroms", pbc=False,
)
hbonds = view.interactions.add("buch", tag="hbonds")
hbonds.set_color("#34d399")
hbonds.set_radius("0.025 nm")
pd.DataFrame(view.interactions.analyses())[
    ["name", "n_structures", "n_evaluated_structures", "n_occurrences"]
]

,name,n_structures,n_evaluated_structures,n_occurrences
0,buch,3,3,65


In Saved sets, try Hide/Show, Inspect, Select participants and Focus participants. Change the structure and inspect again: the participants and distances in nm must belong to the visible frame.

`involving_selection` keeps all interactions involving a selected atom. `within_selection` keeps only interactions entirely within it. `across_selection_boundary` keeps interactions between that selection and the rest; `between_selections` uses explicit disjoint selections A and B.

Filtering a visual set retains the complete scientific analysis. An evaluated structure with no matching observations remains distinct from an unevaluated structure.

In [16]:
hbonds.set_filter(selection=[5], mode="involving_selection")
view.interactions.inspect("hbonds", structure_index=0)

{'tag': 'hbonds',
 'analysis_name': 'buch',
 'analysis_revision': 'sha256:3868b1d6ca83466b9bdcf03322552c24b4557761c953cac31f2fbd64bb9f729d',
 'query_revision': 'sha256:e3abe3ed1db153232af1f49aaa84723029c20ce697c2ee282b22cd55dc1a9c92',
 'frame': 0,
 'status': 'evaluated',
 'method': 'molsysmt.interactions.hbonds.get_buch_hbonds',
 'parameters': {'distance_threshold_nm': 0.39999999999999997,
  'distance_definition': 'hydrogen_acceptor',
  'pbc': False,
  'exclude_self_donor_acceptor': True,
  'role_identification': {'donor_inclusion_rules': ["(atom_type=='O') bonded to (atom_type=='H')",
    "(atom_type=='N') bonded to (atom_type=='H')"],
   'donor_exclusion_rules': ["(atom_name=='NE2') not bonded to (atom_type=='H')",
    "(atom_name=='ND1') not bonded to (atom_type=='H')"],
   'acceptor_inclusion_rules': ["atom_type=='O'",
    "atom_type=='N'",
    "atom_type=='S'"],
   'acceptor_exclusion_rules': ["atom_name=='NE2' and group_name=='GLN'",
    "(atom_name=='NE2' and group_name=='HIS') 

Restore the complete display and go to local frame 1 before saving.

In [17]:
hbonds.set_filter(selection="all", mode="involving_selection")
view.player.go_to_structure(1)

The trajectory preview shows the calculated hydrogen-bond contacts:

<iframe src="../../../_static/views/workbench_interactions.html" title="Pentalanine hydrogen-bond contacts" style="width:100%;height:520px;border:0;" loading="lazy"></iframe>

For selective queries, method choices and independent-file correspondence, continue with {doc}`interactions_workbench`.

## 5. Saving and recovering your work

A **`.msv` session** contains the molecular system, named scientific analyses and visual scene. It can reopen on its own. An **H5MSM interactions file** below contains only the analyses; it is useful when you want to reuse those results with the corresponding molecular system.

All tutorial files are written in a temporary directory. They remain available while `workspace` is alive and are removed when that object is cleaned up or the kernel ends. Copy files you want to keep before ending the session.

In [18]:
session_file = output / "workbench.msv"
interactions_file = output / "interactions.h5msm"
view.save_session(session_file)
view.interactions.save(str(interactions_file), overwrite=True)
print("Session:", session_file.name)
print("Scientific analyses:", interactions_file.name)

Session: workbench.msv
Scientific analyses: interactions.h5msm


Reopen the session as a new view. Compare its structure index, filters, colors, visibility, framing and interactions with the original canvas. Rotate it, navigate to another frame and use Inspect to check that the recovered view remains interactive.

In [19]:
view_C = msv.load_session(session_file, height="520px")
view_C

In [20]:
assert view_C.player.index == view.player.index
saved_set = view.interactions.records()[0]
restored_set = view_C.interactions.get(saved_set["tag"])
assert restored_set.filter == saved_set["filter"]
assert restored_set.style == saved_set["style"]
pd.DataFrame(view_C.interactions.analyses())[
    ["name", "n_structures", "n_evaluated_structures", "n_occurrences"]
]

,name,n_structures,n_evaluated_structures,n_occurrences
0,buch,3,3,65


You can also load one analysis from the H5MSM file into the recovered view, using a different name to preserve the existing analysis. This file was written from the same atom and structure axes, so you can declare `assume_aligned=True`. For an independent file, first establish that correspondence, including explicit remapping if its order differs. Matching dimensions alone do not establish molecular identity.

In [21]:
loaded = view_C.interactions.load(
    str(interactions_file), analysis_name="buch", name="imported-buch",
    assume_aligned=True,
)
assert loaded.n_interactions == result.n_interactions
view_C.interactions.analyses()[-1]["name"]

'imported-buch'

You have used one workflow through both Python and Studio: compose sources, control source regions, keep calculated analyses separate from their visual sets, and recover the complete session. Continue with {doc}`../export/index` to share a visual scene, or {doc}`interactions_workbench` to explore scientific queries and file import.